# FourBi - Single Notebook Demo

Notebook nay chay doc lap de binarize mot anh hoac tat ca anh trong mot thu muc bang checkpoint FourBi.

1. Sua cac duong dan trong cell **Configuration**.
2. Chay tat ca cell tu tren xuong duoi.
3. Ket qua se duoc hien thi va co the luu ra `OUTPUT_DIR`.

> Checkpoint duoc nap voi `weights_only=False` vi checkpoint FourBi co luu config va trang thai training. Chi su dung checkpoint tu nguon tin cay.

In [ ]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

# ========================= CONFIGURATION =========================
# Checkpoint .pth da upload vao Kaggle/Colab hoac nam tren may cua ban.
CHECKPOINT_PATH = Path('ckpts/3d22_HDIBCO18.pth')

# Nhap mot file anh, hoac mot thu muc. Thu muc co the chua anh truc tiep
# hoac theo cau truc dataset cua repo: <folder>/imgs/*.png.
INPUT_PATH = Path('path/to/input_image_or_folder')

# Dat la None neu chi muon visualize; dat Path('output_demo') de luu anh output.
OUTPUT_DIR = None

# None: tu dong tim <dataset>/gt_imgs khi INPUT_PATH la dataset, imgs/, hoac mot anh trong imgs/.
# Neu ground truth nam noi khac, dat vi du Path('/path/to/gt_imgs').
GROUND_TRUTH_DIR = None

PATCH_SIZE = 512
BATCH_SIZE = 8
THRESHOLD = None  # None: lay threshold trong checkpoint; hoac dat vi du 0.5
MAX_DISPLAY = None  # None: visualize tat ca anh; hoac so luong toi da muon xem
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device: {DEVICE}')

## Model definition

Cell nay chua kien truc FourBi can thiet de notebook khong phu thuoc vao cac file `.py` khac trong repository.

In [ ]:
def get_activation(kind='tanh'):
    if kind == 'tanh':
        return nn.Tanh()
    if kind == 'sigmoid':
        return nn.Sigmoid()
    if kind is False:
        return nn.Identity()
    raise ValueError(f'Unknown activation kind: {kind}')


class FourierUnit(nn.Module):
    def __init__(self, in_channels, out_channels, groups=1, spatial_scale_factor=None,
                 spatial_scale_mode='bilinear', spectral_pos_encoding=False,
                 ffc3d=False, fft_norm='ortho'):
        super().__init__()
        self.groups = groups
        self.conv_layer = nn.Conv2d(in_channels * 2 + (2 if spectral_pos_encoding else 0),
                                    out_channels * 2, kernel_size=1, groups=groups, bias=False)
        self.bn = nn.BatchNorm2d(out_channels * 2)
        self.relu = nn.ReLU(inplace=True)
        self.spatial_scale_factor = spatial_scale_factor
        self.spatial_scale_mode = spatial_scale_mode
        self.spectral_pos_encoding = spectral_pos_encoding
        self.ffc3d = ffc3d
        self.fft_norm = fft_norm

    def forward(self, x):
        batch = x.shape[0]
        if self.spatial_scale_factor is not None:
            original_size = x.shape[-2:]
            x = F.interpolate(x, scale_factor=self.spatial_scale_factor,
                              mode=self.spatial_scale_mode, align_corners=False)
        fft_dim = (-3, -2, -1) if self.ffc3d else (-2, -1)
        transformed = torch.fft.rfftn(x, dim=fft_dim, norm=self.fft_norm)
        transformed = torch.stack((transformed.real, transformed.imag), dim=-1)
        transformed = transformed.permute(0, 1, 4, 2, 3).contiguous()
        transformed = transformed.view((batch, -1) + transformed.size()[3:])
        if self.spectral_pos_encoding:
            height, width = transformed.shape[-2:]
            coords_v = torch.linspace(0, 1, height, device=x.device)[None, None, :, None].expand(batch, 1, height, width)
            coords_h = torch.linspace(0, 1, width, device=x.device)[None, None, None, :].expand(batch, 1, height, width)
            transformed = torch.cat((coords_v, coords_h, transformed), dim=1)
        transformed = self.relu(self.bn(self.conv_layer(transformed)))
        transformed = transformed.view((batch, -1, 2) + transformed.size()[2:]).permute(0, 1, 3, 4, 2).contiguous()
        transformed = torch.complex(transformed[..., 0], transformed[..., 1])
        output = torch.fft.irfftn(transformed, s=x.shape[-3:] if self.ffc3d else x.shape[-2:],
                                  dim=fft_dim, norm=self.fft_norm)
        if self.spatial_scale_factor is not None:
            output = F.interpolate(output, size=original_size, mode=self.spatial_scale_mode, align_corners=False)
        return output


class SpectralTransform(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, groups=1, enable_lfu=True, **fu_kwargs):
        super().__init__()
        self.enable_lfu = enable_lfu
        self.downsample = nn.AvgPool2d(2, stride=2) if stride == 2 else nn.Identity()
        self.conv1 = nn.Sequential(nn.Conv2d(in_channels, out_channels // 2, 1, groups=groups, bias=False),
                                   nn.BatchNorm2d(out_channels // 2), nn.ReLU(inplace=True))
        self.fu = FourierUnit(out_channels // 2, out_channels // 2, groups, **fu_kwargs)
        if enable_lfu:
            self.lfu = FourierUnit(out_channels // 2, out_channels // 2, groups)
        self.conv2 = nn.Conv2d(out_channels // 2, out_channels, 1, groups=groups, bias=False)

    def forward(self, x):
        x = self.conv1(self.downsample(x))
        output = self.fu(x)
        if self.enable_lfu:
            _, channels, height, _ = x.shape
            split_size = height // 2
            xs = torch.cat(torch.split(x[:, :channels // 4], split_size, dim=-2), dim=1).contiguous()
            xs = torch.cat(torch.split(xs, split_size, dim=-1), dim=1).contiguous()
            xs = self.lfu(xs).repeat(1, 1, 2, 2).contiguous()
        else:
            xs = 0
        return self.conv2(x + output + xs)


class FFC(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, ratio_gin, ratio_gout,
                 stride=1, padding=0, dilation=1, groups=1, bias=False, enable_lfu=True,
                 padding_type='reflect', gated=False, **spectral_kwargs):
        super().__init__()
        in_cg, out_cg = int(in_channels * ratio_gin), int(out_channels * ratio_gout)
        in_cl, out_cl = in_channels - in_cg, out_channels - out_cg
        self.ratio_gout = ratio_gout
        self.global_in_num = in_cg
        kwargs = dict(kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
                      groups=groups, bias=bias, padding_mode=padding_type)
        self.convl2l = nn.Identity() if in_cl == 0 or out_cl == 0 else nn.Conv2d(in_cl, out_cl, **kwargs)
        self.convl2g = nn.Identity() if in_cl == 0 or out_cg == 0 else nn.Conv2d(in_cl, out_cg, **kwargs)
        self.convg2l = nn.Identity() if in_cg == 0 or out_cl == 0 else nn.Conv2d(in_cg, out_cl, **kwargs)
        self.convg2g = nn.Identity() if in_cg == 0 or out_cg == 0 else SpectralTransform(
            in_cg, out_cg, stride, 1 if groups == 1 else groups // 2, enable_lfu, **spectral_kwargs)
        self.gated = gated
        self.gate = nn.Identity() if in_cg == 0 or out_cl == 0 or not gated else nn.Conv2d(in_channels, 2, 1)

    def forward(self, x):
        x_l, x_g = x if type(x) is tuple else (x, 0)
        if self.gated:
            total = torch.cat([x_l, x_g], dim=1) if torch.is_tensor(x_g) else x_l
            g2l_gate, l2g_gate = torch.sigmoid(self.gate(total)).chunk(2, dim=1)
        else:
            g2l_gate = l2g_gate = 1
        out_l = self.convl2l(x_l) + self.convg2l(x_g) * g2l_gate if self.ratio_gout != 1 else 0
        out_g = self.convl2g(x_l) * l2g_gate + self.convg2g(x_g) if self.ratio_gout != 0 else 0
        return out_l, out_g


class FFC_BN_ACT(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, ratio_gin, ratio_gout,
                 stride=1, padding=0, dilation=1, groups=1, bias=False, norm_layer=nn.BatchNorm2d,
                 activation_layer=nn.Identity, padding_type='reflect', enable_lfu=True, **kwargs):
        super().__init__()
        self.ffc = FFC(in_channels, out_channels, kernel_size, ratio_gin, ratio_gout, stride, padding,
                       dilation, groups, bias, enable_lfu, padding_type=padding_type, **kwargs)
        global_channels = int(out_channels * ratio_gout)
        self.bn_l = nn.Identity() if ratio_gout == 1 else norm_layer(out_channels - global_channels)
        self.bn_g = nn.Identity() if ratio_gout == 0 else norm_layer(global_channels)
        self.act_l = nn.Identity() if ratio_gout == 1 else activation_layer(inplace=True)
        self.act_g = nn.Identity() if ratio_gout == 0 else activation_layer(inplace=True)

    def forward(self, x):
        x_l, x_g = self.ffc(x)
        return self.act_l(self.bn_l(x_l)), self.act_g(self.bn_g(x_g))


class FFCResnetBlock(nn.Module):
    def __init__(self, dim, padding_type, norm_layer, activation_layer=nn.ReLU, dilation=1, inline=False, **kwargs):
        super().__init__()
        self.conv1 = FFC_BN_ACT(dim, dim, 3, padding=dilation, dilation=dilation, norm_layer=norm_layer,
                                activation_layer=activation_layer, padding_type=padding_type, **kwargs)
        self.conv2 = FFC_BN_ACT(dim, dim, 3, padding=dilation, dilation=dilation, norm_layer=norm_layer,
                                activation_layer=activation_layer, padding_type=padding_type, **kwargs)
        self.inline = inline

    def forward(self, x):
        if self.inline:
            x_l, x_g = x[:, :-self.conv1.ffc.global_in_num], x[:, -self.conv1.ffc.global_in_num:]
        else:
            x_l, x_g = x if type(x) is tuple else (x, 0)
        id_l, id_g = x_l, x_g
        x_l, x_g = self.conv1((x_l, x_g))
        x_l, x_g = self.conv2((x_l, x_g))
        result = (id_l + x_l, id_g + x_g)
        return torch.cat(result, dim=1) if self.inline else result


class ConcatTupleLayer(nn.Module):
    def forward(self, x):
        return torch.cat(x, dim=1) if torch.is_tensor(x[1]) else x[0]


class Fourbi(nn.Module):
    def __init__(self, input_nc, output_nc, ngf=64, n_downsampling=3, n_blocks=3,
                 norm_layer=nn.BatchNorm2d, padding_type='reflect', activation_layer=nn.ReLU,
                 up_norm_layer=nn.BatchNorm2d, up_activation=nn.ReLU(True), init_conv_kwargs=None,
                 downsample_conv_kwargs=None, resnet_conv_kwargs=None, add_out_act=True,
                 max_features=1024, out_ffc=False, skip_connections='cat', unet_layers=2):
        super().__init__()
        init_conv_kwargs = init_conv_kwargs or {}
        downsample_conv_kwargs = downsample_conv_kwargs or {}
        resnet_conv_kwargs = resnet_conv_kwargs or {}
        conv3 = dict(kernel_size=3, stride=1, padding=1)
        self.reflect = nn.ReflectionPad2d(3)
        self.skip_connections = skip_connections
        down_channels = [ngf]
        layer = [FFC_BN_ACT(input_nc, ngf, 7, padding=0, norm_layer=norm_layer,
                            activation_layer=activation_layer, **init_conv_kwargs)]
        for _ in range(unet_layers):
            layer.append(FFC_BN_ACT(ngf, ngf, norm_layer=norm_layer, activation_layer=activation_layer,
                                    **init_conv_kwargs, **conv3))
        down_layers = [nn.Sequential(*layer)]
        for index in range(n_downsampling):
            mult = 2 ** index
            kwargs = dict(downsample_conv_kwargs)
            if index == n_downsampling - 1:
                kwargs['ratio_gout'] = resnet_conv_kwargs.get('ratio_gin', 0)
            channels = min(max_features, ngf * mult * 2)
            down_channels.append(channels)
            layer = [FFC_BN_ACT(min(max_features, ngf * mult), channels, 3, stride=2, padding=1,
                                norm_layer=norm_layer, activation_layer=activation_layer, **kwargs)]
            if index < n_downsampling - 1:
                for _ in range(unet_layers):
                    layer.append(FFC_BN_ACT(channels, channels, norm_layer=norm_layer,
                                            activation_layer=activation_layer, **kwargs, **conv3))
            down_layers.append(nn.Sequential(*layer))
        bottleneck = min(max_features, ngf * (2 ** n_downsampling))
        resnet_layers = [FFCResnetBlock(bottleneck, padding_type, norm_layer,
                                         activation_layer=activation_layer, **resnet_conv_kwargs)
                         for _ in range(n_blocks)] + [ConcatTupleLayer()]
        up_layers = []
        for index in range(n_downsampling):
            mult = 2 ** (n_downsampling - index)
            in_channels = min(max_features, ngf * mult)
            in_channels += down_channels.pop() if skip_connections == 'cat' else 0
            channels = min(max_features, int(ngf * mult / 2))
            layer = [nn.ConvTranspose2d(in_channels, channels, 3, stride=2, padding=1, output_padding=1),
                     up_norm_layer(channels), up_activation]
            for _ in range(unet_layers):
                layer.extend([nn.Conv2d(channels, channels, **conv3), nn.BatchNorm2d(channels), nn.ReLU()])
            up_layers.append(nn.Sequential(*layer))
        if out_ffc:
            raise NotImplementedError
        out_channels = ngf + (down_channels.pop() if skip_connections == 'cat' else 0)
        temporary_out = output_nc if unet_layers == 0 else out_channels
        layer = [nn.ReflectionPad2d(3), nn.Conv2d(out_channels, temporary_out, 7)]
        for index in range(unet_layers):
            temporary_out = output_nc if index == unet_layers - 1 else out_channels
            layer.extend([nn.BatchNorm2d(out_channels), nn.ReLU(), nn.Conv2d(out_channels, temporary_out, 3, padding=1)])
        up_layers.append(nn.Sequential(*layer))
        self.final_act = get_activation('tanh' if add_out_act is True else add_out_act)
        self.down_sampling_layers = nn.Sequential(*down_layers)
        self.resnet_layers = nn.Sequential(*resnet_layers)
        self.up_sampling_layers = nn.Sequential(*up_layers)

    def forward(self, x):
        x = self.reflect(x)
        skip_outputs = []
        for layer in self.down_sampling_layers:
            x = layer(x)
            if self.skip_connections != 'none':
                parts = x if torch.is_tensor(x[1]) else (x[0],)
                skip_outputs.append(torch.cat(parts, dim=1))
        x = self.resnet_layers(x)
        for layer in self.up_sampling_layers:
            if self.skip_connections != 'none':
                skip = skip_outputs.pop()
                x = torch.cat([x, skip], dim=1) if self.skip_connections == 'cat' else x + skip
            x = layer(x)
        return self.final_act(x)

## Load checkpoint

In [ ]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Khong tim thay checkpoint: {CHECKPOINT_PATH}')

# FourBi checkpoints contain more than tensor weights; use only a trusted checkpoint.
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
config = checkpoint['config']
model = Fourbi(
    input_nc=config['input_channels'],
    output_nc=config['output_channels'],
    n_downsampling=config['n_downsampling'],
    init_conv_kwargs=config['init_conv_kwargs'],
    downsample_conv_kwargs=config['down_sample_conv_kwargs'],
    resnet_conv_kwargs=config['resnet_conv_kwargs'],
    n_blocks=config['n_blocks'],
    unet_layers=config['unet_layers'],
).to(DEVICE)
model.load_state_dict(checkpoint['model'], strict=True)
model.eval()
threshold = config.get('threshold', 0.5) if THRESHOLD is None else THRESHOLD

print(f'Loaded checkpoint: {CHECKPOINT_PATH.name}')
print(f'Patch size: {PATCH_SIZE}, batch size: {BATCH_SIZE}, threshold: {threshold}')

## Inference utilities

In [ ]:
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}


def find_images(path):
    path = Path(path)
    if path.is_file():
        if path.suffix.lower() not in IMAGE_EXTENSIONS:
            raise ValueError(f'File khong phai dinh dang anh ho tro: {path}')
        return [path]
    if not path.is_dir():
        raise FileNotFoundError(f'Khong tim thay input: {path}')
    search_dir = path / 'imgs' if (path / 'imgs').is_dir() else path
    paths = sorted(p for p in search_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    if not paths:
        raise FileNotFoundError(f'Khong tim thay anh trong: {search_dir}')
    return paths


def find_ground_truth(image_path):
    if GROUND_TRUTH_DIR is not None:
        gt_dir = Path(GROUND_TRUTH_DIR)
    elif image_path.parent.name == 'imgs' and (image_path.parent.parent / 'gt_imgs').is_dir():
        gt_dir = image_path.parent.parent / 'gt_imgs'
    else:
        return None
    exact_match = gt_dir / image_path.name
    if exact_match.exists():
        return exact_match
    for suffix in IMAGE_EXTENSIONS:
        candidate = gt_dir / f'{image_path.stem}{suffix}'
        if candidate.exists():
            return candidate
    return None


def image_to_tensor(image):
    array = np.asarray(image.convert('RGB'), dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1)


def make_patches(image, patch_size):
    stride = patch_size // 2
    tensor = image_to_tensor(image).unsqueeze(0)
    _, _, height, width = tensor.shape
    padding_bottom = ((height // patch_size) + 1) * patch_size - height
    padding_right = ((width // patch_size) + 1) * patch_size - width
    tensor = F.pad(tensor, (0, padding_right, 0, padding_bottom), value=1.0)
    unfolded = tensor.unfold(2, patch_size, stride).unfold(3, patch_size, stride)
    n_rows, n_cols = unfolded.shape[2], unfolded.shape[3]
    patches = unfolded.permute(0, 2, 3, 1, 4, 5).reshape(-1, 3, patch_size, patch_size)
    return patches, n_rows, n_cols, stride, (height, width)


def stitch_predictions(predictions, n_rows, n_cols, stride, original_size):
    patch_size = predictions.shape[-1]
    padded_height = patch_size + (n_rows - 1) * stride
    padded_width = patch_size + (n_cols - 1) * stride
    patches = predictions.reshape(n_rows, n_cols, 1, patch_size, patch_size)
    canvas = torch.zeros(1, padded_height, padded_width)
    x_steps = [x + stride // 2 for x in range(0, padded_height, stride)]
    y_steps = [y + stride // 2 for y in range(0, padded_width, stride)]
    x_steps[0], x_steps[-1] = 0, padded_height
    y_steps[0], y_steps[-1] = 0, padded_width
    for row in range(n_rows):
        for col in range(n_cols):
            x1, x2 = x_steps[row], x_steps[row + 1]
            y1, y2 = y_steps[col], y_steps[col + 1]
            px1, px2 = x1 - row * stride, x2 - row * stride
            py1, py2 = y1 - col * stride, y2 - col * stride
            canvas[:, x1:x2, y1:y2] = patches[row, col, :, px1:px2, py1:py2]
    height, width = original_size
    return canvas[:, :height, :width]


@torch.inference_mode()
def binarize_image(image_path):
    source = Image.open(image_path).convert('RGB')
    patches, n_rows, n_cols, stride, size = make_patches(source, PATCH_SIZE)
    predictions = []
    for batch in torch.split(patches, BATCH_SIZE):
        predictions.append(model(batch.to(DEVICE)).cpu())
    prediction = stitch_predictions(torch.cat(predictions), n_rows, n_cols, stride, size)
    binary = (prediction > threshold).to(torch.uint8).squeeze(0).numpy() * 255
    return source, Image.fromarray(binary, mode='L')


def calculate_psnr(output, ground_truth):
    prediction = np.asarray(output.convert('L'), dtype=np.float32) / 255.0
    target = np.asarray(ground_truth.convert('L'), dtype=np.float32) / 255.0
    if prediction.shape != target.shape:
        raise ValueError(f'Kich thuoc output va ground truth khong trung: {prediction.shape} vs {target.shape}')
    target = (target > threshold).astype(np.float32)
    mse = np.mean((prediction - target) ** 2)
    return float('inf') if mse == 0 else float(10 * np.log10(1.0 / mse))


def visualize_results(results, max_display=None):
    visible = results if max_display is None else results[:max_display]
    for start in range(0, len(visible), 5):
        group = visible[start:start + 5]
        figure, axes = plt.subplots(len(group), 2, figsize=(14, 4 * len(group)), squeeze=False)
        for row, (path, source, output, psnr) in enumerate(group):
            axes[row, 0].imshow(source)
            axes[row, 0].set_title(f'Input: {path.name}')
            axes[row, 1].imshow(output, cmap='gray', vmin=0, vmax=255)
            score_text = 'GT not found' if psnr is None else f'PSNR: {psnr:.4f} dB'
            axes[row, 1].set_title(f'FourBi output - {score_text}')
            axes[row, 0].axis('off')
            axes[row, 1].axis('off')
        plt.tight_layout()
        plt.show()

## Run demo and visualize output

In [ ]:
image_paths = find_images(INPUT_PATH)
results = []

if OUTPUT_DIR is not None:
    OUTPUT_DIR = Path(OUTPUT_DIR)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

psnr_scores = []
for index, path in enumerate(image_paths, start=1):
    source, output = binarize_image(path)
    gt_path = find_ground_truth(path)
    psnr = None
    if gt_path is not None:
        ground_truth = Image.open(gt_path).convert('L')
        psnr = calculate_psnr(output, ground_truth)
        psnr_scores.append(psnr)
    results.append((path, source, output, psnr))
    if OUTPUT_DIR is not None:
        output.save(OUTPUT_DIR / f'{path.stem}.png')
    metric_text = 'GT not found' if psnr is None else f'PSNR: {psnr:.4f} dB'
    print(f'[{index}/{len(image_paths)}] Processed: {path.name} - {metric_text}')

visualize_results(results, MAX_DISPLAY)
print(f'Done. Visualized {len(results) if MAX_DISPLAY is None else min(len(results), MAX_DISPLAY)} image(s).')
if psnr_scores:
    print(f'Mean PSNR over {len(psnr_scores)} image(s): {np.mean(psnr_scores):.4f} dB')
else:
    print('Khong tim thay ground truth; khong the tinh PSNR.')
if OUTPUT_DIR is not None:
    print(f'Saved outputs to: {OUTPUT_DIR}')